# Файнтюн bi-encoder'а на train.parquet (Avito, кандидатогенерация)

Идея: `ai-forever/ru-en-RoSBERTa` — общая модель, не заточенная под жаргон объявлений услуг
Авито ("обзвон по базе", "холодный обзвон клиентов" и т.п.). У нас есть 497 673 пары
`(search_query → выбранный item)` из `train.parquet` — это готовый датасет для
контрастивного файнтюна bi-encoder'а под конкретно эту задачу и лексику.

**Метод:** `MultipleNegativesRankingLoss` (sentence-transformers) — стандартный лосс для
сетапа "позитивная пара + остальные примеры батча как in-batch негативы", ровно то, что нужно
для retrieval-задачи.

**Важно про честность оценки:**
- Split делаем **по уникальному тексту запроса** (`search_query`), а не по строкам/парам.
  Так гарантируем, что запросы в валидации модель не видела на файнтюне вообще —
  иначе получим оптимистичную оценку (data leakage).
- Корпус для eval **того же размера, что в реальном бенчмарке (~189k)**: ground truth
  val-запросов + «отвлекающие» объявления из `benchmark_items.parquet` (реальное
  распределение корпуса). Иначе поиск в корпусе из 30k в ~6 раз проще и Recall@50
  завышен.
- **Hold-out по объявлениям** (`ITEM_HOLDOUT=True`): объявления, которые являются
  ground truth для val-запросов, вырезаются из train-пар — модель не видела их даже
  по другим запросам. Это консервативная (нижняя) оценка; см. диагностику пересечения
  benchmark_items и train в ячейке корпуса.
- Recall@50 на этом ноутбуке — всё ещё **прокси** (другие запросы, чем в
  benchmark_queries, и релевантными считаются только клики из train), но масштаб корпуса теперь честный. Она нужна, чтобы честно
  сравнить "до" и "после" файнтюна при одинаковых условиях, а не как финальная оценка
  решения — финальную оценку всё равно считает `solution.py --validate` на реальных файлах.

**Что дальше:** сохранённую модель можно подставить прямо в `solution.py` через
`--semantic-model /path/to/rosberta-avito-finetuned` (SentenceTransformer грузит
локальную папку так же, как имя модели с HuggingFace).


## 0. Установка зависимостей

Если уже установлены — ячейку можно пропустить.

## 1. Конфиг

In [ ]:
import gc
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# --- пути к данным (поменяйте под себя) ---
TRAIN_PATH = "train.parquet"

# --- базовая модель, которую дообучаем ---
BASE_MODEL_NAME = "intfloat/multilingual-e5-small"
USE_PREFIXES = True                    # RoSBERTa обучалась с префиксами
QUERY_PREFIX = "search_query: "
DOC_PREFIX = "search_document: "

# --- как строим тексты объявления/запроса---
DESC_CHARS = 300                       # semantic-канал выигрывает от доп. контекста (в отличие от tfidf)
INFM_PARAMS_TRUNC = 500

# --- split ---
VAL_QUERY_FRACTION = 0.03              # доля УНИКАЛЬНЫХ текстов запроса, уходящая в held-out val
MAX_PAIRS_PER_QUERY_FOR_TRAIN = 20     # защита от переобучения на супер-частые запросы

# --- размер eval-корпуса (объявления, среди которых ищем top-50) ---
EVAL_CORPUS_SIZE = 189212         

# --- обучение ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32 if DEVICE == "cuda" else 16
GRAD_ACCUM_STEPS = 4
USE_FP16 = DEVICE == "cuda"    
USE_GRADIENT_CHECKPOINTING = True   
EPOCHS = 2
WARMUP_STEPS = 1000
ENCODE_BATCH_SIZE = 64 if DEVICE == "cuda" else 16
TOP_K = 50
OUTPUT_MODEL_DIR = "rosberta-avito-finetuned"

print(f"device = {DEVICE}, batch_size = {BATCH_SIZE}, grad_accum = {GRAD_ACCUM_STEPS} "
      f"(эффективный батч = {BATCH_SIZE * GRAD_ACCUM_STEPS}), fp16 = {USE_FP16}")

# --- честная оценка ---
BENCHMARK_ITEMS_PATH = "benchmark_items.parquet"  # источник «отвлекающих» объявлений
MAX_VAL_QUERIES = 3000       # ~ как число запросов бенчмарка (2452); лишние уходят в train
ITEM_HOLDOUT = True          # вырезать val-ground-truth объявления из train-пар
import os

RANK_QUERY_FRACTION = 0.10   
MAX_RANK_QUERIES = 15000


device = cuda, batch_size = 32, grad_accum = 4 (эффективный батч = 128), fp16 = True


## 2. Загрузка train.parquet и построение текстов

Тексты строим так же, как в `solution.py` для semantic-канала: заголовок + infm_params
(+ немного описания), с префиксами `search_query:` / `search_document:`.


In [3]:
import re

def clean_light(s):
    """Лёгкая чистка: только схлопывание пробелов, без нижнего регистра и т.п.
    Плотным эмбеддингам не нужна агрессивная нормализация, как char-TF-IDF."""
    if s is None:
        return ""
    return re.sub(r"\s+", " ", str(s)).strip()

def build_query_text(query, infm_params):
    q = clean_light(query)
    p = clean_light(infm_params)
    text = (q + ". " + p).strip(". ")
    return (QUERY_PREFIX + text) if USE_PREFIXES else text

def build_item_text(title, infm_params, description=None):
    t = clean_light(title)
    p = clean_light(infm_params)[:INFM_PARAMS_TRUNC]
    text = (t + ". " + p).strip(". ")
    if DESC_CHARS > 0 and description:
        text = (text + ". " + clean_light(description)[:DESC_CHARS]).strip(". ")
    return (DOC_PREFIX + text) if USE_PREFIXES else text


print("Загрузка train.parquet...")
t0 = time.time()
cols = [
    "search_query", "search_infm_params_text",
    "item_id", "item_title_raw", "item_infm_params_text", "item_description_raw",
]
train_df = pd.read_parquet(TRAIN_PATH, columns=cols, engine="pyarrow")
print(f"  строк: {len(train_df)}, время: {time.time()-t0:.1f}s")

train_df["query_text"] = [
    build_query_text(q, p)
    for q, p in zip(train_df["search_query"], train_df["search_infm_params_text"])
]
train_df["item_text"] = [
    build_item_text(t, p, d)
    for t, p, d in zip(
        train_df["item_title_raw"],
        train_df["item_infm_params_text"],
        train_df["item_description_raw"],
    )
]
print(f"  тексты построены, время: {time.time()-t0:.1f}s")
train_df.head(3)


Загрузка train.parquet...
  строк: 497673, время: 4.4s
  тексты построены, время: 35.7s


,search_query,search_infm_params_text,item_id,item_title_raw,item_infm_params_text,item_description_raw,query_text,item_text
0,скупка телевизоров,,e8b685dffe1a408e,Скупка б/у техники,Вид услуги Место оказания услуг Первомайский п...,Скупаю практически любую современную новую и б...,search_query: скупка телевизоров,search_document: Скупка б/у техники. Вид услуг...
1,автоподбор,Рейтинг пользователя 4 звезды и выше,92e1b0446f827b59,Автоподбор Разовый осмотр автомобиля,Вид услуги Место оказания услуг Нижний Новгоро...,🚗 Автоподбор и выездная диагностика автомобиля...,search_query: автоподбор. Рейтинг пользователя...,search_document: Автоподбор Разовый осмотр авт...
2,баня на дровах,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",624846856ce81d69,"Баня на дровах ""Прованс"" на Цветочной","Вид услуги Красота, здоровье Место оказания ус...",В ритме современной жизни так сложно найти мо...,search_query: баня на дровах. Онлайн-запись Ти...,"search_document: Баня на дровах ""Прованс"" на Ц..."


## 3. Дедупликация пар и split без утечек

- Дедуп по `(search_query, item_id)` — одинаковые клики не должны учитываться много раз
  как будто это разные примеры (иначе оверфит на пару "частый запрос → частый item").
- Split **по уникальному сырому тексту запроса** (`search_query`), а не по строкам:
  так ни один pair с val-запросом не попадает в train, и val честно проверяет обобщение
  на невиданные запросы (а не просто невиданные пары).


In [ ]:
# --- дедуп пар (query, item) ---
before = len(train_df)
train_df = train_df.drop_duplicates(subset=["search_query", "item_id"]).reset_index(drop=True)
print(f"Пар до дедупа: {before}, после дедупа: {len(train_df)}")

# --- считаем частоту каждого запроса, чтобы отрезать длинный хвост супер-частых запросов ---
query_counts = train_df["search_query"].value_counts()
print(f"Уникальных текстов запроса: {len(query_counts)}")
print(query_counts.describe())

# --- split уникальных ЗАПРОСОВ на train/val (не строк!) ---
unique_queries = query_counts.index.to_numpy()
rng = np.random.RandomState(RANDOM_SEED)
rng.shuffle(unique_queries)
n_val_queries = max(1, int(len(unique_queries) * VAL_QUERY_FRACTION))
val_queries = set(unique_queries[:n_val_queries])

n_val_queries = min(n_val_queries, MAX_VAL_QUERIES)
val_queries = set(unique_queries[:n_val_queries])

n_rank_queries = min(int(len(unique_queries) * RANK_QUERY_FRACTION), MAX_RANK_QUERIES)
rank_queries = set(unique_queries[n_val_queries:n_val_queries + n_rank_queries])
train_queries = set(unique_queries[n_val_queries + n_rank_queries:])

import json
with open("heldout_queries.json", "w", encoding="utf-8") as f:
    json.dump({"val_queries": sorted(val_queries), "rank_queries": sorted(rank_queries)},
              f, ensure_ascii=False)
print(f"Сохранён heldout_queries.json: val={len(val_queries)}, rank={len(rank_queries)}")

train_pairs = train_df[train_df["search_query"].isin(train_queries)].reset_index(drop=True)
val_pairs = train_df[train_df["search_query"].isin(val_queries)].reset_index(drop=True)

print(f"\nTrain: {len(train_pairs)} пар / {len(train_queries)} уникальных запросов")
print(f"Val:   {len(val_pairs)} пар / {len(val_queries)} уникальных запросов")
assert train_queries.isdisjoint(val_queries) and train_queries.isdisjoint(rank_queries)  # финальная страховка от утечки


Пар до дедупа: 497673, после дедупа: 445261
Уникальных текстов запроса: 74529
count    74529.000000
mean         5.974332
std         50.302240
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max       5474.000000
Name: count, dtype: float64
Сохранён heldout_queries.json: val=2235, rank=7452

Train: 396665 пар / 64842 уникальных запросов
Val:   11396 пар / 2235 уникальных запросов


In [5]:
# --- hold-out по объявлениям: выбрасываем из train пары с item_id, которые являются
#     ground truth для val-запросов. Иначе модель «видела» искомое объявление на файнтюне
#     (по другому запросу), и Recall@50 завышен. ---
val_gt_ids = set(val_pairs["item_id"])
if ITEM_HOLDOUT:
    n0 = len(train_pairs)
    train_pairs = train_pairs[~train_pairs["item_id"].isin(val_gt_ids)].reset_index(drop=True)
    print(f"ITEM_HOLDOUT: удалено {n0 - len(train_pairs)} train-пар с val-объявлениями")

# --- режем длинный хвост: для каждого train-запроса оставляем не больше
#     MAX_PAIRS_PER_QUERY_FOR_TRAIN пар (иначе супер-частые запросы доминируют в батчах
#     и модель просто переобучится на них, а не научится обобщать) ---
def cap_per_group(df, group_col, cap, rng):
    parts = []
    for _, g in df.groupby(group_col, sort=False):
        if len(g) > cap:
            g = g.sample(n=cap, random_state=rng)
        parts.append(g)
    return pd.concat(parts, ignore_index=True)

train_pairs = cap_per_group(train_pairs, "search_query", MAX_PAIRS_PER_QUERY_FOR_TRAIN, rng)
print(f"Train-пар после обрезки хвоста: {len(train_pairs)}")


ITEM_HOLDOUT: удалено 7893 train-пар с val-объявлениями
Train-пар после обрезки хвоста: 190044


## 4. Корпус для валидационного retrieval-теста (размер как в бенчмарке)

Корпус = (1) все ground-truth объявления val-запросов + (2) «отвлекающие» объявления из
`benchmark_items.parquet` (реальный корпус соревнования) до размера `EVAL_CORPUS_SIZE`
(189 212). Если файла нет — запасной вариант: отвлекающие из train (корпус может оказаться
меньше, тогда оценка будет завышена; в ячейке есть предупреждение).

Дополнительно печатаем, какая доля benchmark_items вообще встречается в train: если она
высокая, ITEM_HOLDOUT=True слишком пессимистичен, и истинный результат лежит между
двумя вариантами.


In [6]:
val_gt_items = val_pairs[["item_id", "item_text"]].drop_duplicates(subset=["item_id"])
print(f"Уникальных ground-truth объявлений в val: {len(val_gt_items)}")

if os.path.exists(BENCHMARK_ITEMS_PATH):
    bench = pd.read_parquet(
        BENCHMARK_ITEMS_PATH,
        columns=["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw"],
    )
    # тексты строим ровно так же, как для train (те же функции и параметры)
    bench["item_text"] = [
        build_item_text(t, p, d)
        for t, p, d in zip(bench["item_title_raw"], bench["item_infm_params_text"],
                           bench["item_description_raw"])
    ]
    # диагностика: насколько объявления бенчмарка пересекаются с train
    overlap = bench["item_id"].isin(set(train_df["item_id"])).mean()
    print(f"Доля benchmark_items, встречающихся в train: {overlap:.1%}")
    distractor_pool = bench[["item_id", "item_text"]].drop_duplicates(subset=["item_id"])
    del bench
else:
    print("ВНИМАНИЕ: benchmark_items.parquet не найден — берём отвлекающие из train.")
    distractor_pool = train_df[["item_id", "item_text"]].drop_duplicates(subset=["item_id"])

# отвлекающие не должны дублировать ground truth
distractor_pool = distractor_pool[~distractor_pool["item_id"].isin(val_gt_items["item_id"])]

n_distractors = max(0, EVAL_CORPUS_SIZE - len(val_gt_items))
distractors = distractor_pool.sample(n=min(n_distractors, len(distractor_pool)), random_state=RANDOM_SEED)

eval_corpus = pd.concat([val_gt_items, distractors], ignore_index=True).drop_duplicates(subset=["item_id"])
eval_corpus = eval_corpus.reset_index(drop=True)
print(f"Итоговый размер eval-корпуса: {len(eval_corpus)} (бенчмарк: 189212)")
if len(eval_corpus) < 0.9 * 189212:
    print("ВНИМАНИЕ: корпус заметно меньше бенчмарка — Recall@50 будет завышен.")

corpus_item_ids = eval_corpus["item_id"].to_numpy()
corpus_texts = eval_corpus["item_text"].tolist()
item_id_to_corpus_idx = {iid: i for i, iid in enumerate(corpus_item_ids)}

del distractor_pool, distractors
gc.collect()


Уникальных ground-truth объявлений в val: 11277
Доля benchmark_items, встречающихся в train: 9.6%
Итоговый размер eval-корпуса: 189212 (бенчмарк: 189212)


0

## 5. Функция оценки Recall@50

Одна и та же функция используется "до" и "после" файнтюна — чтобы сравнение было честным
(тот же корпус, тот же набор val-запросов, та же метрика, что в `solution.py`).


In [7]:
def top_k_idx(row, k=TOP_K):
    top_idx = np.argpartition(-row, k)[:k]
    return top_idx[np.argsort(-row[top_idx])]

def evaluate_recall_at_k(model, val_pairs_df, corpus_texts, corpus_item_ids,
                          item_id_to_corpus_idx, k=TOP_K, label=""):
    """Recall@k: для каждого запроса из val_pairs_df считаем долю подходящих item_id,
    попавших в top-k. У нас в val_pairs может быть несколько строк на один и тот же
    query_text (несколько кликов по одному запросу) — группируем их, как в реальной
    задаче, где на запрос может быть несколько релевантных объявлений."""
    print(f"\n--- Оценка: {label} ---")
    t0 = time.time()

    # эмбеддинги корпуса считаем один раз
    corpus_emb = model.encode(
        corpus_texts, batch_size=ENCODE_BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)

    # группируем val по уникальному query_text -> множество релевантных item_id
    grouped = val_pairs_df.groupby("query_text")["item_id"].apply(set)
    query_texts = grouped.index.tolist()
    relevant_sets = grouped.tolist()

    query_emb = model.encode(
        query_texts, batch_size=ENCODE_BATCH_SIZE, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)

    recalls = []
    BATCH = 256
    for start in range(0, len(query_texts), BATCH):
        end = min(start + BATCH, len(query_texts))
        sim = query_emb[start:end] @ corpus_emb.T
        for i in range(end - start):
            top_idx = top_k_idx(sim[i])
            top_item_ids = set(corpus_item_ids[top_idx])
            rel = relevant_sets[start + i]
            # учитываем только релевантные item_id, которые вообще есть в eval-корпусе
            rel_in_corpus = {iid for iid in rel if iid in item_id_to_corpus_idx}
            if not rel_in_corpus:
                continue
            hit = len(rel_in_corpus & top_item_ids) / len(rel_in_corpus)
            recalls.append(hit)

    recall = float(np.mean(recalls))
    print(f"  Recall@{k} = {recall:.4f}  (запросов: {len(recalls)}, время: {time.time()-t0:.1f}s)")

    del corpus_emb, query_emb
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return recall


## 6. Baseline: оценка ДО файнтюна

In [8]:
from sentence_transformers import SentenceTransformer

print(f"Загружаем базовую модель '{BASE_MODEL_NAME}'...")
base_model = SentenceTransformer(BASE_MODEL_NAME, device=DEVICE)

recall_before = evaluate_recall_at_k(
    base_model, val_pairs, corpus_texts, corpus_item_ids, item_id_to_corpus_idx,
    label="БАЗОВАЯ модель (до файнтюна)",
)

del base_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()



Загружаем базовую модель 'ai-forever/ru-en-RoSBERTa'...


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



--- Оценка: БАЗОВАЯ модель (до файнтюна) ---


Batches:   0%|          | 0/2957 [00:00<?, ?it/s]

KeyboardInterrupt: 

## 7. Файнтюн

`MultipleNegativesRankingLoss`: в батче из B пар `(query_i, item_i)` для каждого `query_i`
позитив — это `item_i`, а все остальные `item_j` (`j != i`) в батче автоматически становятся
негативами. Чем больше батч — тем "жёстче" негативы и обычно лучше качество, но больше памяти GPU.

Без изменения набора префиксов относительно инференса: если модель обучена с
`search_query:` / `search_document:`, используем те же префиксы при последующем инференсе
в `solution.py` (`--semantic-model` уже поддерживает произвольный путь к модели, а
`use_semantic_prefix=True` там включён по умолчанию — оставьте так же).


In [ ]:
from datasets import Dataset
from sentence_transformers import losses
from sentence_transformers.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.trainer import SentenceTransformerTrainer

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    gc.collect()
    print("Свободно на GPU перед загрузкой модели:",
          f"{torch.cuda.mem_get_info()[0] / 1e9:.2f} GB из "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB")

print(f"Загружаем '{BASE_MODEL_NAME}' для дообучения (поверх, не с нуля)...")
model = SentenceTransformer(BASE_MODEL_NAME, device=DEVICE)

if USE_GRADIENT_CHECKPOINTING:
    # включаем checkpointing на внутреннем HF-трансформере (экономит активации)
    try:
        model[0].auto_model.gradient_checkpointing_enable()
        print("Gradient checkpointing включён.")
    except Exception as e:
        print(f"Не удалось включить gradient checkpointing: {e}")

# Trainer из sentence-transformers v3+ ожидает datasets.Dataset, а не InputExample —
# для MultipleNegativesRankingLoss достаточно двух колонок: (anchor, positive).
train_dataset = Dataset.from_dict({
    "query": train_pairs["query_text"].tolist(),
    "item": train_pairs["item_text"].tolist(),
})
print(f"Обучающих примеров: {len(train_dataset)}")

train_loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_MODEL_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    fp16=USE_FP16,                # смешанная точность — вдвое меньше памяти
    report_to="none",             # отключаем tensorboard/wandb
    save_strategy="no",           # без промежуточных чекпоинтов
    logging_steps=100,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
)

t0 = time.time()
try:
    trainer.train()
except torch.cuda.OutOfMemoryError:
    print(
        "\nOOM даже с текущими настройками. Дальнейшие шаги:\n"
        "1. Kernel -> Restart, затем заново прогнать ячейки конфига и данных.\n"
        "2. В конфиге ещё уменьшить BATCH_SIZE (например, до 8 или 16) и/или\n"
        "   увеличить GRAD_ACCUM_STEPS, чтобы сохранить эффективный размер батча.\n"
        "3. Проверить в отдельном терминале `nvidia-smi` — не занимает ли GPU\n"
        "   другой процесс (например, зависшее прошлое ядро Jupyter)."
    )
    raise
print(f"Файнтюн завершён за {(time.time()-t0)/60:.1f} мин")

# явное сохранение финальной модели (это НЕ то же самое, что чекпоинты Trainer'а выше)
model.save(OUTPUT_MODEL_DIR)
print(f"Модель сохранена в: {OUTPUT_MODEL_DIR}")


## 8. Оценка ПОСЛЕ файнтюна и сравнение

In [ ]:
recall_after = evaluate_recall_at_k(
    model, val_pairs, corpus_texts, corpus_item_ids, item_id_to_corpus_idx,
    label="ДООБУЧЕННАЯ модель (после файнтюна)",
)

print("\n" + "=" * 50)
print(f"Recall@{TOP_K} до файнтюна:    {recall_before:.4f}")
print(f"Recall@{TOP_K} после файнтюна: {recall_after:.4f}")
delta = recall_after - recall_before
print(f"Изменение:               {delta:+.4f} ({delta / max(recall_before, 1e-9):+.1%})")
print("=" * 50)
print(
    f"\nОценка на корпусе из {len(corpus_texts)} объявлений (бенчмарк: 189212), "
    f"{'с' if ITEM_HOLDOUT else 'без'} hold-out по объявлениям. Это консервативная оценка "
    "по held-out запросам; финальную цифру всё равно даёт solution.py --validate."
)


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 4))
bars = ax.bar(["до файнтюна", "после файнтюна"], [recall_before, recall_after],
              color=["#9aa5b1", "#2b6cb0"])
ax.set_ylabel(f"Recall@{TOP_K} (held-out, прокси)")
ax.set_ylim(0, max(recall_before, recall_after) * 1.3)
for b in bars:
    h = b.get_height()
    ax.annotate(f"{h:.3f}", (b.get_x() + b.get_width() / 2, h),
                ha="center", va="bottom")
plt.tight_layout()
plt.show()
